# J-space / Jacobian Lens Stage 2 bounded discrimination (DESIGN: NOT AUTHORIZED)

Purpose: measure whether the pinned Qwen3-1.7B Jacobian-lens readout carries
reproducible structure not recoverable from five cheaper baselines, at the exact
Stage 1 loci. This is Stage 2 (observational discrimination) per
`EvoScientist/skills/jspace-research-operations/SKILL.md`.

**PROVISIONAL: EXECUTION NOT AUTHORIZED.** This notebook implements the Stage 2
discrimination proposal. The preregistered thresholds are Dr. Mani's provisional
defaults; they are NOT ratified. Do NOT run any cell on a GPU until Dr. Mani
ratifies the open questions (see the final stage-gate cell).

Boundaries (carried forward from Stage 1, unchanged):
- Observation only; no lens fitting, steering, ablation, activation edit,
  Sakshi/Elume mutation, or external publication.
- Only the versioned synthetic, non-sensitive stimulus manifest below is used.
- Store per-prompt digests and token/byte counts, not raw prompt text, in every
  exported observation artifact.
- Lens readouts are measurements (evidence class 1), not ground truth about
  beliefs, intentions, consciousness, or cognition. No promotion to class 2/3.
- The preflight aborts before model/lens downloads if CUDA or minimum VRAM is
  unavailable. Insufficient hardware is a measured result, not permission to
  weaken the gate.

In Colab, select **Runtime -> Change runtime type -> GPU (T4)**, then, only after
ratification, run cells top-to-bottom.


## 0. Pinned source identities (copied verbatim from the Stage 1 smoke test)

These revisions and checksums are copied exactly from the known-good Stage 1
smoke test. Stage 2 introduces no new model or lens and re-resolves and asserts
every identity at execution time.


In [ ]:
# --- Copied verbatim from jspace_colab_smoke_test.ipynb cell 0 ---
JLENS_REPO_URL = "https://github.com/anthropics/jacobian-lens.git"
JLENS_COMMIT = "581d398613e5602a5af361e1c34d3a92ea82ba8e"
MODEL_ID = "Qwen/Qwen3-1.7B"
MODEL_REVISION = "70d244cc86ccca08cf5af4e1e306ecf908b1ad5e"
LENS_REPO = "neuronpedia/jacobian-lens"
LENS_REVISION = "a4114d7752d11eb546e6cf372213d7e75526d3a1"
LENS_FILE = "qwen3-1.7b/jlens/Salesforce-wikitext/Qwen3-1.7B_jacobian_lens.pt"
MIN_VRAM_GIB = 14.0
TOP_K = 10
MAX_PROMPT_TOKENS = 128

# --- Stage 2 additions: expected artifact identity (asserted at execution) ---
EXPECTED_MODEL_D_MODEL = 2048
EXPECTED_MODEL_N_LAYERS = 28
EXPECTED_LENS_SIZE_BYTES = 226501315
EXPECTED_LENS_SHA256 = "6fcc79011bd921ffd87612255e2e99950a124fa519470ee44ebaf161c39be9d6"

# --- Stage 2 fixed measurement loci (Section 3 of the proposal) ---
SELECTED_LAYERS = [6, 13, 20, 26]   # fixed to the Stage 1 loci; no free-parameter search
POSITIONS = [-2]                     # fixed to the Stage 1 position
TOP_N = 50                           # rank statistics computed over top-N, only sparse summaries retained

# --- Stage 1 reproduction anchor (kill criterion) ---
STAGE1_PROMPT_SHA256 = "daeaa63881dc0f58be689307a81b1fbc347674424f1cae45819f82372804f5a6"

DISCRIMINATION_SCHEMA = "jspace-observation-discrimination/v1"


## 0b. Preregistered thresholds (PROVISIONAL: pending Dr. Mani ratification)

These constants lock the Section 6 criteria BEFORE any data is collected. They
are Dr. Mani's adopted provisional defaults and are NOT yet ratified. No
threshold may be changed after data collection begins. `THRESHOLDS_RATIFIED`
must be flipped to `True` by Dr. Mani before the measurement cells run; the
export records this flag.


In [ ]:
# PROVISIONAL preregistered thresholds. Do NOT edit after data collection begins.
# Ratification gate: measurement is refused unless Dr. Mani sets this True.
THRESHOLDS_RATIFIED = False

# Specificity (condition 1): D(Jacobian,real vs control) must exceed
# D(Jacobian, logit-lens) by at least SPECIFICITY_D_MARGIN, on at least
# SPECIFICITY_MIN_PROMPT_FRACTION of prompts, for BOTH control families.
SPECIFICITY_MIN_PROMPT_FRACTION = 0.80
SPECIFICITY_D_MARGIN = 0.10   # provisional; requires Dr. Mani ratification

# Added information (condition 2): median top-10 Jaccard between Jacobian and
# logit-lens must be <= ADDED_INFO_MEDIAN_JACCARD_MAX, with a paired Wilcoxon
# signed-rank test at WILCOXON_ALPHA over per-prompt D values; and the Jacobian
# readout must not be within rerun noise of the output or prompt-only baselines.
ADDED_INFO_MEDIAN_JACCARD_MAX = 0.70
WILCOXON_ALPHA = 0.01

# Repeats and seeds (Section 7).
SAME_RUNTIME_REPEATS = 2
INFERENCE_SEEDS = [0, 1]        # seed 0 as in Stage 1 + one control seed for seed-invariance
RANDOM_VECTOR_SEEDS = [0, 1, 2]  # the random-vector baseline is the only stochastic component

# Same-runtime rerun noise reference measured in Stage 1 (max abs logit diff).
STAGE1_RERUN_NOISE_MAX_ABS_LOGIT_DIFF = 0.0

PROVISIONAL_THRESHOLDS = {
    "specificity_min_prompt_fraction": SPECIFICITY_MIN_PROMPT_FRACTION,
    "specificity_d_margin": SPECIFICITY_D_MARGIN,
    "added_info_median_jaccard_max": ADDED_INFO_MEDIAN_JACCARD_MAX,
    "wilcoxon_alpha": WILCOXON_ALPHA,
    "same_runtime_repeats": SAME_RUNTIME_REPEATS,
    "inference_seeds": INFERENCE_SEEDS,
    "random_vector_seeds": RANDOM_VECTOR_SEEDS,
    "stage1_rerun_noise_max_abs_logit_diff": STAGE1_RERUN_NOISE_MAX_ABS_LOGIT_DIFF,
    "ratified": THRESHOLDS_RATIFIED,
}


## 1. Install the pinned reference implementation

The Git commit is pinned. Model and lens artifacts are separately pinned in the
configuration cell. `scipy` provides the paired Wilcoxon test and Spearman
correlation used in the discrimination metrics.


In [ ]:
%pip install -q "git+https://github.com/anthropics/jacobian-lens.git@{JLENS_COMMIT}" "transformers>=5.5" "huggingface_hub>=0.30" "safetensors" "scipy>=1.10"


## 2. Fail-fast accelerator and environment preflight

This runs before downloading the model or lens. A failure is a measured capacity
result; do not weaken the threshold merely to force the experiment through.


In [ ]:
import json
import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path

import torch
import transformers
import huggingface_hub
import scipy

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU unavailable. In Colab select Runtime -> Change runtime type -> GPU.")

device_index = torch.cuda.current_device()
props = torch.cuda.get_device_properties(device_index)
vram_gib = props.total_memory / (1024 ** 3)
if vram_gib < MIN_VRAM_GIB:
    raise RuntimeError(
        f"Assigned GPU has {vram_gib:.2f} GiB VRAM; this bounded study requires at least "
        f"{MIN_VRAM_GIB:.1f} GiB. Request another runtime or move to an L4-class VM."
    )

compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
disk = shutil.disk_usage("/content" if Path("/content").exists() else ".")
try:
    nvidia_smi = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total,driver_version,compute_cap", "--format=csv,noheader"],
        check=True, capture_output=True, text=True,
    ).stdout.strip()
except Exception as exc:
    nvidia_smi = f"unavailable: {type(exc).__name__}: {exc}"

environment = {
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "huggingface_hub": huggingface_hub.__version__,
    "scipy": scipy.__version__,
    "cuda_runtime": torch.version.cuda,
    "gpu_name": props.name,
    "gpu_total_vram_gib": round(vram_gib, 3),
    "compute_capability": f"{props.major}.{props.minor}",
    "compute_dtype": str(compute_dtype),
    "disk_free_gib": round(disk.free / (1024 ** 3), 3),
    "nvidia_smi": nvidia_smi,
}
print(json.dumps(environment, indent=2))


## 3. Resolve and verify exact Hub revisions and lens checksum

This checks that the requested revisions resolve to the expected commits,
downloads only the lens file, computes its SHA-256 before deserialization, and
asserts the lens byte size and checksum equal the Stage 1 known-good values.


In [ ]:
import hashlib
from huggingface_hub import HfApi, hf_hub_download

api = HfApi()
resolved_model_revision = api.model_info(MODEL_ID, revision=MODEL_REVISION).sha
resolved_lens_revision = api.model_info(LENS_REPO, revision=LENS_REVISION).sha
assert resolved_model_revision == MODEL_REVISION, (resolved_model_revision, MODEL_REVISION)
assert resolved_lens_revision == LENS_REVISION, (resolved_lens_revision, LENS_REVISION)

lens_path = hf_hub_download(
    repo_id=LENS_REPO,
    filename=LENS_FILE,
    revision=LENS_REVISION,
)

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

lens_size_bytes = os.path.getsize(lens_path)
lens_sha256 = sha256_file(lens_path)

# Stage 2 kill criterion: the pinned lens must resolve to the exact recorded identity.
assert lens_size_bytes == EXPECTED_LENS_SIZE_BYTES, (lens_size_bytes, EXPECTED_LENS_SIZE_BYTES)
assert lens_sha256 == EXPECTED_LENS_SHA256, (lens_sha256, EXPECTED_LENS_SHA256)

print(json.dumps({
    "model_revision": resolved_model_revision,
    "lens_revision": resolved_lens_revision,
    "lens_file": LENS_FILE,
    "lens_size_bytes": lens_size_bytes,
    "lens_sha256": lens_sha256,
}, indent=2))


## 4. Load the matching model and lens; verify compatibility and transport primitives

The compatibility checks are mandatory: model width, layer count, lens width,
and pinned identity must agree. Stage 2 additionally asserts the fixed loci are
inside the fitted source layers, and probes for the low-level lens transport
primitives the random-vector and structure-broken baselines require. If those
primitives are named differently in the pinned jlens build, this cell fails fast
with an explicit blocker rather than fabricating a readout.


In [ ]:
import jlens
from transformers import AutoModelForCausalLM, AutoTokenizer

jlens.configure_logging()
torch.manual_seed(0)
torch.cuda.manual_seed_all(0)

hf_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    dtype=compute_dtype,
    low_cpu_mem_usage=True,
).cuda().eval()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
model = jlens.from_hf(hf_model, tokenizer, compile=False)
lens = jlens.JacobianLens.load(lens_path)

assert model.d_model == EXPECTED_MODEL_D_MODEL, f"Unexpected model width: {model.d_model}"
assert model.n_layers == EXPECTED_MODEL_N_LAYERS, f"Unexpected layer count: {model.n_layers}"
assert lens.d_model == model.d_model, (lens.d_model, model.d_model)
assert lens.source_layers, "Lens contains no fitted source layers"
assert min(lens.source_layers) >= 0 and max(lens.source_layers) < model.n_layers

# Stage 2: the fixed loci must be inside the fitted source layers.
missing_loci = [layer for layer in SELECTED_LAYERS if layer not in set(lens.source_layers)]
assert not missing_loci, f"Selected loci not in fitted source layers: {missing_loci}"

# Capability probe for the raw-vector transport used by the random-vector and
# structure-broken baselines. These attribute names are the documented ASSUMPTION
# to confirm against jlens commit 581d398 before ratified execution; if the build
# exposes different names, adjust here and re-pin. Fail fast instead of guessing.
def resolve_transport_fn(lens_obj):
    for name in ("transport", "transport_activation", "apply_jacobian", "push"):
        fn = getattr(lens_obj, name, None)
        if callable(fn):
            return name, fn
    return None, None

def resolve_jacobian_accessor(lens_obj):
    for name in ("jacobian", "get_jacobian", "source_jacobian"):
        fn = getattr(lens_obj, name, None)
        if callable(fn):
            return name, fn
    for name in ("jacobians", "source_jacobians"):
        mapping = getattr(lens_obj, name, None)
        if mapping is not None:
            return name, mapping
    return None, None

TRANSPORT_NAME, TRANSPORT_FN = resolve_transport_fn(lens)
JAC_NAME, JAC_ACCESSOR = resolve_jacobian_accessor(lens)
if TRANSPORT_FN is None and JAC_ACCESSOR is None:
    raise NotImplementedError(
        "BLOCKER: the pinned jlens build exposes no recognized raw-vector transport "
        "primitive (tried transport/transport_activation/apply_jacobian/push and "
        "jacobian/get_jacobian/jacobians). The random-vector and structure-broken "
        "baselines require transporting an arbitrary activation through a chosen "
        "source-layer Jacobian. Inspect the jlens API at commit "
        f"{JLENS_COMMIT} and update resolve_transport_fn / resolve_jacobian_accessor, "
        "then re-pin. Do not substitute a different transport silently."
    )

print(model)
print(lens)
print(json.dumps({
    "transport_primitive": TRANSPORT_NAME,
    "jacobian_accessor": JAC_NAME,
    "selected_layers": SELECTED_LAYERS,
    "source_layers": lens.source_layers,
}, indent=2))
print(f"allocated={torch.cuda.memory_allocated() / (1024**3):.3f} GiB; "
      f"reserved={torch.cuda.memory_reserved() / (1024**3):.3f} GiB")


## 5. Versioned synthetic stimulus manifest (50 prompts, 5 categories x 10)

Synthetic, non-sensitive, English, short. Item 1 of category 1 is the exact
Stage 1 prompt so its reproduction is checked inside this run (kill criterion).
Raw text lives only in this manifest; observation artifacts store digests and
counts only. Per-prompt SHA-256 and token counts are computed at runtime, and
the Stage 1 prompt digest is asserted against its pinned value.


In [ ]:
STIMULUS_MANIFEST_VERSION = "jspace-stage2-stimulus/v1"

# (category, prompt). Category 1, item 1 is the exact Stage 1 prompt.
RAW_STIMULI = [
    ("factual_completion", "Fact: The currency used in the country shaped like a boot is"),
    ("factual_completion", "Fact: The capital city of France is"),
    ("factual_completion", "Fact: The chemical symbol for gold is"),
    ("factual_completion", "Fact: The largest planet in our solar system is"),
    ("factual_completion", "Fact: The author of Romeo and Juliet is"),
    ("factual_completion", "Fact: The tallest mountain on Earth is"),
    ("factual_completion", "Fact: The primary language spoken in Brazil is"),
    ("factual_completion", "Fact: The number of continents on Earth is"),
    ("factual_completion", "Fact: The metal that is liquid at room temperature is"),
    ("factual_completion", "Fact: The ocean between America and Europe is the"),
    ("category_membership", "A robin is a kind of"),
    ("category_membership", "A hammer is a kind of"),
    ("category_membership", "A rose is a kind of"),
    ("category_membership", "A salmon is a kind of"),
    ("category_membership", "A violin is a kind of"),
    ("category_membership", "A maple is a kind of"),
    ("category_membership", "Copper is a kind of"),
    ("category_membership", "A triangle is a kind of"),
    ("category_membership", "A sedan is a kind of"),
    ("category_membership", "Oxygen is a kind of"),
    ("arithmetic_completion", "Compute: 2 + 3 ="),
    ("arithmetic_completion", "Compute: 7 + 5 ="),
    ("arithmetic_completion", "Compute: 10 - 4 ="),
    ("arithmetic_completion", "Compute: 6 * 3 ="),
    ("arithmetic_completion", "Compute: 9 + 9 ="),
    ("arithmetic_completion", "Compute: 12 - 7 ="),
    ("arithmetic_completion", "Compute: 4 * 4 ="),
    ("arithmetic_completion", "Compute: 15 + 6 ="),
    ("arithmetic_completion", "Compute: 20 - 8 ="),
    ("arithmetic_completion", "Compute: 3 * 7 ="),
    ("antonym_negation", "The opposite of hot is"),
    ("antonym_negation", "The opposite of up is"),
    ("antonym_negation", "The opposite of fast is"),
    ("antonym_negation", "The opposite of happy is"),
    ("antonym_negation", "The opposite of open is"),
    ("antonym_negation", "The opposite of light is"),
    ("antonym_negation", "The opposite of empty is"),
    ("antonym_negation", "The opposite of true is"),
    ("antonym_negation", "The opposite of early is"),
    ("antonym_negation", "The opposite of soft is"),
    ("multi_token_entity", "The novel begins: It was the best of times, it was the worst of"),
    ("multi_token_entity", "The scientist Albert"),
    ("multi_token_entity", "The city of New"),
    ("multi_token_entity", "The composer Ludwig van"),
    ("multi_token_entity", "The painter Vincent van"),
    ("multi_token_entity", "The river known as the Mississippi flows into the Gulf of"),
    ("multi_token_entity", "The statue in New York harbor is the Statue of"),
    ("multi_token_entity", "The company founded by Steve Jobs is called"),
    ("multi_token_entity", "The wall that once divided Berlin was the Berlin"),
    ("multi_token_entity", "The telescope launched in 1990 is the Hubble Space"),
]
assert len(RAW_STIMULI) == 50, len(RAW_STIMULI)

stimulus_manifest = []
for index, (category, text) in enumerate(RAW_STIMULI):
    text_bytes = text.encode("utf-8")
    prompt_sha256 = hashlib.sha256(text_bytes).hexdigest()
    token_count = int(tokenizer(text, return_tensors="pt", truncation=True,
                                max_length=MAX_PROMPT_TOKENS).input_ids.shape[-1])
    stimulus_manifest.append({
        "id": f"s{index:02d}",
        "index": index,
        "category": category,
        "text": text,
        "sha256": prompt_sha256,
        "utf8_byte_count": len(text_bytes),
        "token_count": token_count,
    })

# Kill criterion: the Stage 1 prompt must reproduce its pinned digest.
assert stimulus_manifest[0]["sha256"] == STAGE1_PROMPT_SHA256, (
    stimulus_manifest[0]["sha256"], STAGE1_PROMPT_SHA256)

manifest_doc = {
    "manifest_version": STIMULUS_MANIFEST_VERSION,
    "n_prompts": len(stimulus_manifest),
    "categories": sorted({row["category"] for row in stimulus_manifest}),
    "prompts": stimulus_manifest,
}
manifest_bytes = (json.dumps(manifest_doc, sort_keys=True, indent=2, ensure_ascii=False) + "\n").encode("utf-8")
STIMULUS_MANIFEST_SHA256 = hashlib.sha256(manifest_bytes).hexdigest()
print(json.dumps({
    "manifest_version": STIMULUS_MANIFEST_VERSION,
    "n_prompts": len(stimulus_manifest),
    "categories": manifest_doc["categories"],
    "stimulus_manifest_sha256": STIMULUS_MANIFEST_SHA256,
}, indent=2, ensure_ascii=False))


## 6. Compute the six readouts, baselines, and discrimination metrics

For every prompt, at every fixed locus, the notebook computes six directly
comparable top-k readouts in the model's vocabulary basis: the Jacobian lens and
the five baselines (logit-lens, output, prompt-only, norm-matched random-vector,
structure-broken). Full-vocabulary logits are computed transiently and reduced
to sparse top-k plus scalar summaries before being discarded; raw activations,
raw prompts, and full logits are never persisted. Same-runtime repeats (x2) and
the seed set confirm determinism; the random-vector baseline uses its own seeds.

This cell REFUSES to run unless `THRESHOLDS_RATIFIED` is True.


In [ ]:
from datetime import datetime, timezone
from uuid import uuid4
from scipy.stats import spearmanr, wilcoxon

if not THRESHOLDS_RATIFIED:
    raise RuntimeError(
        "STAGE GATE: THRESHOLDS_RATIFIED is False. Stage 2 execution is not "
        "authorized until Dr. Mani ratifies the provisional preregistered "
        "thresholds and the open questions. Do not flip this flag to force a run."
    )

RUN_ID = str(uuid4())


def sparse_topk(logits_row, k=TOP_K):
    values, indices = logits_row.topk(k)
    return [
        {"rank": rank, "token_id": int(token_id),
         "token_text": tokenizer.decode([int(token_id)]), "logit": float(value)}
        for rank, (token_id, value) in enumerate(zip(indices, values), start=1)
    ]


def topk_ids(logits_row, k=TOP_K):
    return [int(i) for i in logits_row.topk(k).indices]


def jaccard_top10(a_row, b_row):
    a = set(topk_ids(a_row, TOP_K))
    b = set(topk_ids(b_row, TOP_K))
    union = a | b
    return len(a & b) / len(union) if union else 1.0


def spearman_topn(a_row, b_row, n=TOP_N):
    union = list(dict.fromkeys(topk_ids(a_row, n) + topk_ids(b_row, n)))
    a_vals = [float(a_row[i]) for i in union]
    b_vals = [float(b_row[i]) for i in union]
    if len(union) < 3:
        return float("nan")
    rho, _ = spearmanr(a_vals, b_vals)
    return float(rho)


def output_argmax_rank(readout_row, output_row):
    target = int(output_row.argmax())
    order = [int(i) for i in readout_row.argsort(descending=True)]
    return order.index(target) + 1


def logit_diff_summary(a_row, b_row):
    diff = (a_row - b_row).abs()
    return {"max_abs": float(diff.max()), "mean_abs": float(diff.mean())}


def divergence_D(a_row, b_row):
    # Primary per-prompt divergence: complement of top-10 Jaccard blended with
    # normalized mean rank displacement of shared tokens. Larger D = more divergence.
    a_ids = topk_ids(a_row, TOP_K)
    b_ids = topk_ids(b_row, TOP_K)
    jacc = jaccard_top10(a_row, b_row)
    a_rank = {tid: r for r, tid in enumerate(a_ids)}
    b_rank = {tid: r for r, tid in enumerate(b_ids)}
    shared = set(a_ids) & set(b_ids)
    if shared:
        disp = sum(abs(a_rank[t] - b_rank[t]) for t in shared) / (len(shared) * (TOP_K - 1))
    else:
        disp = 1.0
    return 0.5 * (1.0 - jacc) + 0.5 * disp


# --- residual-stream capture via forward hooks on the HF decoder layers ---
def capture_layer_residuals(prompt, layers, position):
    captured = {}
    handles = []

    def make_hook(layer_idx):
        def hook(_module, _inputs, output):
            hidden = output[0] if isinstance(output, tuple) else output
            captured[layer_idx] = hidden[0, position, :].detach().float()
        return hook

    for layer_idx in layers:
        handles.append(hf_model.model.layers[layer_idx].register_forward_hook(make_hook(layer_idx)))
    try:
        enc = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_PROMPT_TOKENS)
        enc = {k: v.cuda() for k, v in enc.items()}
        with torch.inference_mode():
            hf_model(**enc)
    finally:
        for handle in handles:
            handle.remove()
    return captured


def decode_residual(residual_vec):
    # Direct (logit-lens-style) decode of a residual vector through the model's
    # final norm and unembedding. Used for the prompt-only baseline and to decode
    # transported random / structure-broken vectors into the same vocab basis.
    vec = residual_vec.to(hf_model.dtype)
    normed = hf_model.model.norm(vec)
    return hf_model.lm_head(normed).float().cpu()


def input_embedding_residual(prompt, position):
    enc = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_PROMPT_TOKENS)
    input_ids = enc.input_ids.cuda()
    with torch.inference_mode():
        embeds = hf_model.get_input_embeddings()(input_ids)
    return embeds[0, position, :].detach()


def get_layer_jacobian(source_layer):
    if callable(JAC_ACCESSOR):
        return JAC_ACCESSOR(source_layer)
    if JAC_ACCESSOR is not None:
        return JAC_ACCESSOR[source_layer]
    return None


def transport_vector(activation_vec, source_layer):
    # Transport an arbitrary residual vector through the chosen source-layer
    # Jacobian, using whichever primitive the load cell resolved. Returns logits.
    if callable(TRANSPORT_FN):
        transported = TRANSPORT_FN(activation_vec, source_layer)
        # transport may already return vocab logits or a residual; decode if residual-shaped.
        if transported.shape[-1] == model.d_model:
            return decode_residual(transported)
        return transported.float()
    jac = get_layer_jacobian(source_layer)
    transported = jac.to(activation_vec.dtype) @ activation_vec
    if transported.shape[-1] == model.d_model:
        return decode_residual(transported)
    return transported.float()


def random_vector_readout(activation_vec, source_layer, seed):
    generator = torch.Generator(device=activation_vec.device).manual_seed(seed)
    noise = torch.randn(activation_vec.shape, generator=generator,
                        device=activation_vec.device, dtype=torch.float32)
    scaled = noise * (float(activation_vec.float().norm()) / float(noise.norm()))
    return transport_vector(scaled.to(activation_vec.dtype), source_layer)


READOUT_PAIRS = [
    ("jacobian", "logit_lens"),
    ("jacobian", "output"),
    ("jacobian", "prompt_only"),
    ("jacobian", "random_vector"),
    ("jacobian", "non_jspace_shuffled"),
    ("jacobian", "non_jspace_mismatched"),
]

per_prompt_artifacts = []
paired_D = {pair: [] for pair in READOUT_PAIRS}
paired_D_control_vs_logit = {"random_vector": [], "non_jspace_shuffled": [], "non_jspace_mismatched": []}

for row in stimulus_manifest:
    prompt = row["text"]
    position = POSITIONS[0]

    with torch.inference_mode():
        j1, output_logits_1, observed_ids = lens.apply(
            model, prompt, layers=SELECTED_LAYERS, positions=POSITIONS, max_seq_len=MAX_PROMPT_TOKENS)
        logit_lens, _, _ = lens.apply(
            model, prompt, layers=SELECTED_LAYERS, positions=POSITIONS,
            max_seq_len=MAX_PROMPT_TOKENS, use_jacobian=False)
        j2, output_logits_2, _ = lens.apply(
            model, prompt, layers=SELECTED_LAYERS, positions=POSITIONS, max_seq_len=MAX_PROMPT_TOKENS)

    residuals = capture_layer_residuals(prompt, SELECTED_LAYERS, position)
    prompt_only_row = decode_residual(input_embedding_residual(prompt, position))

    jacobian_readouts, logit_readouts = {}, {}
    output_readouts, prompt_only_readouts = {}, {}
    random_vector_readouts, non_jspace_readouts = {}, {}
    discrimination = {}
    repeat_same_ids, repeat_max_diff = True, 0.0

    for layer in SELECTED_LAYERS:
        key = str(layer)
        j_row = j1[layer][0]
        ll_row = logit_lens[layer][0]
        out_row = output_logits_1[0]
        po_row = prompt_only_row[0] if prompt_only_row.dim() > 1 else prompt_only_row

        jacobian_readouts[key] = sparse_topk(j_row)
        logit_readouts[key] = sparse_topk(ll_row)
        output_readouts[key] = sparse_topk(out_row)
        prompt_only_readouts[key] = sparse_topk(po_row)

        # Random-vector baseline across its own seed set (summary statistics only).
        rv_seed_rows = {}
        rv_summaries = {}
        for seed in RANDOM_VECTOR_SEEDS:
            rv_logits = random_vector_readout(residuals[layer], layer, seed)
            rv_row = rv_logits[0] if rv_logits.dim() > 1 else rv_logits
            rv_seed_rows[seed] = rv_row
            rv_summaries[str(seed)] = {
                "top_k": sparse_topk(rv_row),
                "D_vs_jacobian_real": divergence_D(j_row, rv_row),
                "jaccard_top10_vs_jacobian_real": jaccard_top10(j_row, rv_row),
            }
        random_vector_readouts[key] = rv_summaries

        # Structure-broken baselines: shuffled-layer and mismatched-probe.
        other_layers = [l for l in SELECTED_LAYERS if l != layer]
        shuffled_source = other_layers[0] if other_layers else layer
        shuffled_logits = transport_vector(residuals[shuffled_source], layer)
        shuffled_row = shuffled_logits[0] if shuffled_logits.dim() > 1 else shuffled_logits
        mismatched_source_layer = other_layers[-1] if other_layers else layer
        mismatched_logits = transport_vector(residuals[layer], mismatched_source_layer)
        mismatched_row = mismatched_logits[0] if mismatched_logits.dim() > 1 else mismatched_logits
        non_jspace_readouts[key] = {
            "shuffled_layer": {
                "applied_source_layer_for_activation": shuffled_source,
                "top_k": sparse_topk(shuffled_row),
                "D_vs_jacobian_real": divergence_D(j_row, shuffled_row),
            },
            "mismatched_probe": {
                "applied_jacobian_of_layer": mismatched_source_layer,
                "top_k": sparse_topk(mismatched_row),
                "D_vs_jacobian_real": divergence_D(j_row, mismatched_row),
            },
        }

        # Discrimination metrics per readout-pair at this locus.
        pair_rows = {
            "logit_lens": ll_row, "output": out_row, "prompt_only": po_row,
            "random_vector": rv_seed_rows[RANDOM_VECTOR_SEEDS[0]],
            "non_jspace_shuffled": shuffled_row, "non_jspace_mismatched": mismatched_row,
        }
        locus_metrics = {}
        for _, other_name in READOUT_PAIRS:
            other_row = pair_rows[other_name]
            locus_metrics[other_name] = {
                "jaccard_top10": jaccard_top10(j_row, other_row),
                "spearman_topN": spearman_topn(j_row, other_row),
                "output_argmax_rank_in_jacobian": output_argmax_rank(j_row, out_row),
                "output_argmax_rank_in_other": output_argmax_rank(other_row, out_row),
                "logit_diff": logit_diff_summary(j_row, other_row),
                "D": divergence_D(j_row, other_row),
            }
        discrimination[key] = locus_metrics

        # Determinism / same-runtime repeat.
        if not j1[layer][0].topk(TOP_K).indices.equal(j2[layer][0].topk(TOP_K).indices):
            repeat_same_ids = False
        repeat_max_diff = max(repeat_max_diff, float((j1[layer] - j2[layer]).abs().max()))

    # Per-prompt paired D (aggregate over loci by mean) for cross-prompt tests.
    for pair in READOUT_PAIRS:
        _, other_name = pair
        paired_D[pair].append(
            sum(discrimination[str(l)][other_name]["D"] for l in SELECTED_LAYERS) / len(SELECTED_LAYERS))
    for control in paired_D_control_vs_logit:
        d_ctrl = sum(discrimination[str(l)][control]["D"] for l in SELECTED_LAYERS) / len(SELECTED_LAYERS)
        paired_D_control_vs_logit[control].append(d_ctrl)

    repeatability = {
        "same_topk_token_ids": repeat_same_ids,
        "max_abs_jacobian_logit_difference": repeat_max_diff,
        "max_abs_model_logit_difference": float((output_logits_1 - output_logits_2).abs().max()),
        "repeats": SAME_RUNTIME_REPEATS,
    }

    observation = {
        "schema": DISCRIMINATION_SCHEMA,
        "artifact_type": "per_prompt",
        "run_id": RUN_ID,
        "observation_id": str(uuid4()),
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "evidence_class": "direct_runtime_measurement",
        "scope": "open_loop_observation_only",
        "model": {"repo_id": MODEL_ID, "revision": MODEL_REVISION,
                  "n_layers": model.n_layers, "d_model": model.d_model},
        "lens": {"repo_id": LENS_REPO, "revision": LENS_REVISION, "filename": LENS_FILE,
                 "sha256": lens_sha256, "n_prompts_recorded_in_artifact": lens.n_prompts,
                 "source_layers": lens.source_layers, "d_model": lens.d_model},
        "instrumentation": {"repo": JLENS_REPO_URL, "commit": JLENS_COMMIT},
        "input": {"sha256": row["sha256"], "utf8_byte_count": row["utf8_byte_count"],
                  "token_count": int(observed_ids.shape[-1]),
                  "raw_prompt_persisted": False, "position_indices": POSITIONS},
        "stimulus": {"id": row["id"], "category": row["category"],
                     "stimulus_manifest_sha256": STIMULUS_MANIFEST_SHA256,
                     "manifest_version": STIMULUS_MANIFEST_VERSION},
        "runtime": environment,
        "measurement": {
            "selected_layers": SELECTED_LAYERS,
            "top_k": TOP_K,
            "top_n": TOP_N,
            "positions": POSITIONS,
            "jacobian_lens": jacobian_readouts,
            "logit_lens_baseline": logit_readouts,
            "output_baseline": output_readouts,
            "prompt_only_baseline": prompt_only_readouts,
            "random_vector_baseline": random_vector_readouts,
            "non_jspace_baseline": non_jspace_readouts,
            "repeatability_same_runtime": repeatability,
            "calibration": "uncalibrated raw logits and ranks",
            "uncertainty": "No construct-level uncertainty model; do not treat readouts as ground truth.",
            "coverage": "Fixed Stage 1 loci and one token position only; not full-sequence or full-layer coverage.",
        },
        "discrimination": {
            "readout_pairs": ["|".join(p) for p in READOUT_PAIRS],
            "per_locus": discrimination,
            "primary_metric": "D = 0.5*(1 - jaccard_top10) + 0.5*normalized_shared_rank_displacement",
        },
        "retention": {"raw_activations_persisted": False, "full_logits_persisted": False,
                      "raw_prompt_persisted": False},
    }
    per_prompt_artifacts.append(observation)

# --- Cross-prompt aggregation (paired nonparametric tests + effect sizes) ---
def paired_stats(values, null=0.0):
    n = len(values)
    median = sorted(values)[n // 2] if n else float("nan")
    try:
        stat, pvalue = wilcoxon([v - null for v in values])
        stat, pvalue = float(stat), float(pvalue)
    except ValueError:
        stat, pvalue = float("nan"), float("nan")
    return {"n": n, "median": median, "mean": (sum(values) / n if n else float("nan")),
            "wilcoxon_stat": stat, "wilcoxon_p": pvalue}

added_info_jaccard = []
for artifact in per_prompt_artifacts:
    vals = [artifact["discrimination"]["per_locus"][str(l)]["logit_lens"]["jaccard_top10"]
            for l in SELECTED_LAYERS]
    added_info_jaccard.append(sum(vals) / len(vals))

median_jaccard_jac_vs_logit = sorted(added_info_jaccard)[len(added_info_jaccard) // 2]

# Specificity check: per prompt, D(jacobian,control) - D(jacobian,logit) >= margin.
def specificity_fraction(control):
    d_logit = paired_D[("jacobian", "logit_lens")]
    d_ctrl = paired_D[("jacobian", control)]
    hits = sum(1 for dc, dl in zip(d_ctrl, d_logit) if (dc - dl) >= SPECIFICITY_D_MARGIN)
    return hits / len(d_ctrl) if d_ctrl else 0.0

specificity = {
    "random_vector_fraction": specificity_fraction("random_vector"),
    "non_jspace_shuffled_fraction": specificity_fraction("non_jspace_shuffled"),
    "non_jspace_mismatched_fraction": specificity_fraction("non_jspace_mismatched"),
}
specificity_pass = (
    specificity["random_vector_fraction"] >= SPECIFICITY_MIN_PROMPT_FRACTION
    and specificity["non_jspace_shuffled_fraction"] >= SPECIFICITY_MIN_PROMPT_FRACTION
    and specificity["non_jspace_mismatched_fraction"] >= SPECIFICITY_MIN_PROMPT_FRACTION
)

added_info_wilcoxon = paired_stats(added_info_jaccard, null=1.0)
added_info_pass = (
    median_jaccard_jac_vs_logit <= ADDED_INFO_MEDIAN_JACCARD_MAX
    and added_info_wilcoxon["wilcoxon_p"] == added_info_wilcoxon["wilcoxon_p"]  # not NaN
    and added_info_wilcoxon["wilcoxon_p"] < WILCOXON_ALPHA
)

reproduction_pass = all(
    a["measurement"]["repeatability_same_runtime"]["same_topk_token_ids"]
    and a["measurement"]["repeatability_same_runtime"]["max_abs_jacobian_logit_difference"]
        <= STAGE1_RERUN_NOISE_MAX_ABS_LOGIT_DIFF
    for a in per_prompt_artifacts if a["stimulus"]["id"] == "s00"
)

if reproduction_pass and specificity_pass and added_info_pass:
    decision = "pass"
elif not reproduction_pass:
    decision = "kill"
elif specificity_pass or added_info_pass:
    decision = "ambiguity"
else:
    decision = "fail"

per_category = {}
for artifact in per_prompt_artifacts:
    cat = artifact["stimulus"]["category"]
    j = sum(artifact["discrimination"]["per_locus"][str(l)]["logit_lens"]["jaccard_top10"]
            for l in SELECTED_LAYERS) / len(SELECTED_LAYERS)
    per_category.setdefault(cat, []).append(j)
per_category_median = {c: sorted(v)[len(v) // 2] for c, v in per_category.items()}

aggregate = {
    "schema": DISCRIMINATION_SCHEMA,
    "artifact_type": "aggregate",
    "run_id": RUN_ID,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "evidence_class": "direct_runtime_measurement",
    "scope": "open_loop_observation_only",
    "model": per_prompt_artifacts[0]["model"],
    "lens": per_prompt_artifacts[0]["lens"],
    "instrumentation": per_prompt_artifacts[0]["instrumentation"],
    "runtime": environment,
    "stimulus_manifest": {
        "sha256": STIMULUS_MANIFEST_SHA256,
        "manifest_version": STIMULUS_MANIFEST_VERSION,
        "n_prompts": len(stimulus_manifest),
        "categories": manifest_doc["categories"],
    },
    "thresholds": PROVISIONAL_THRESHOLDS,
    "provisional_thresholds_ratified": THRESHOLDS_RATIFIED,
    "aggregate": {
        "n_prompts": len(per_prompt_artifacts),
        "median_top10_jaccard_jacobian_vs_logit_lens": median_jaccard_jac_vs_logit,
        "added_information": added_info_wilcoxon,
        "specificity": specificity,
        "per_category_median_jaccard_jacobian_vs_logit_lens": per_category_median,
        "reproduction_pass": reproduction_pass,
        "specificity_pass": specificity_pass,
        "added_information_pass": added_info_pass,
    },
    "decision": {
        "result": decision,
        "notes": "pass/ambiguity/fail/kill per Section 6; provisional thresholds, not ratified.",
    },
    "retention": {"raw_activations_persisted": False, "full_logits_persisted": False,
                  "raw_prompt_persisted": False},
}

print(json.dumps({
    "run_id": RUN_ID,
    "n_prompts": len(per_prompt_artifacts),
    "median_jaccard_jacobian_vs_logit_lens": median_jaccard_jac_vs_logit,
    "specificity": specificity,
    "decision": decision,
}, indent=2, ensure_ascii=False))


## 7. Export content-addressed observation artifacts

Per-prompt observations and the single aggregate run artifact are written as
canonical JSON with filenames derived from their contents. Existing files are
never overwritten with different bytes. Each artifact is read back and re-hashed.
All artifacts use schema `jspace-observation-discrimination/v1` and carry the
retention flags proving raw prompt text, raw activations, and full logits were
not persisted.


In [ ]:
artifact_dir = Path("/content/jspace_artifacts_stage2" if Path("/content").exists() else "jspace_artifacts_stage2")
artifact_dir.mkdir(parents=True, exist_ok=True)


def write_content_addressed(obj, prefix):
    payload = (json.dumps(obj, sort_keys=True, indent=2, ensure_ascii=False) + "\n").encode("utf-8")
    digest = hashlib.sha256(payload).hexdigest()
    path = artifact_dir / f"{prefix}_{digest[:16]}.json"
    try:
        with path.open("xb") as handle:
            handle.write(payload)
    except FileExistsError:
        if sha256_file(path) != digest:
            raise RuntimeError(f"Content-addressed path exists with a different checksum: {path}")
    assert sha256_file(path) == digest
    return path, digest


written = []
for observation in per_prompt_artifacts:
    path, digest = write_content_addressed(observation, "jspace_observation")
    written.append({"path": str(path), "sha256": digest, "stimulus_id": observation["stimulus"]["id"]})

aggregate_path, aggregate_digest = write_content_addressed(aggregate, "jspace_discrimination")

print(json.dumps({
    "n_per_prompt_artifacts": len(written),
    "per_prompt_dir": str(artifact_dir),
    "aggregate_path": str(aggregate_path),
    "aggregate_sha256": aggregate_digest,
    "first_three": written[:3],
}, indent=2, ensure_ascii=False))


## 8. Optional explicit download

Run this only if you are authorized to transfer the derived sparse artifacts from
the Colab runtime to the local computer. Transfer is a separate authorization
requiring independent local hashing; this notebook does not authorize it.


In [ ]:
# Explicitly opt in by changing this to True.
DOWNLOAD_ARTIFACTS = False
if DOWNLOAD_ARTIFACTS:
    from google.colab import files
    for record in written:
        files.download(record["path"])
    files.download(str(aggregate_path))
else:
    print("Artifacts retained in the current runtime only:", str(artifact_dir))


## Stage gate: EXECUTION NOT AUTHORIZED until ratification

This notebook is a Stage 2 design deliverable. Running any measurement cell is
**not authorized** until Dr. Mani ratifies the open questions and the
provisional preregistered thresholds. The measurement cell refuses to run while
`THRESHOLDS_RATIFIED` is False, and flipping that flag is Dr. Mani's decision,
not a workaround.

Ratification checklist (open questions Q1–Q8 from the proposal):
1. Sample size and categories (default: 5 categories x 10 = 50 prompts).
2. Measurement loci (default: fixed Stage 1 loci, layers 6/13/20/26, position -2).
3. Downstream criterion (default: the model's own next-token output).
4. Preregistered thresholds (the PROVISIONAL constants above).
5. Raw stimulus retention (default: versioned in-repo manifest; artifacts stay digest-only).
6. Runtime scope (default: single T4 class only).
7. Validator (default: extend `scripts/validate_observation.py` with the discrimination mode).
8. Execution authorization (default: authoring allowed; GPU execution still pending).

A Stage 2 pass, ambiguous, fail, or kill outcome are all valid, reportable
results. None promotes any readout from evidence class 1 to class 2 or 3, and
none authorizes causal intervention (Stage 3), publication, transfer, or
Sakshi/Elume integration.
